# Current ecotype-imputation baseline

This notebook resolves the latest immutable sightings release, reads the fitted current model, independently reconciles its encounter metrics against its out-of-fold predictions, and exports a compact baseline report. It does **not** refit or promote a model and writes only to `outputs/current/`.

The four evaluation products have different meanings: reconstruction is optimistic context reuse, encounter holds out complete encounters, blocked holds out date/spatial blocks, and purged blocked is a no-local-support stress test.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from experiment_support import resolve_release_paths, write_current_model_outputs

paths = resolve_release_paths()
output_dir = Path(os.environ['MARINE_MAMMALS_RESEARCH_OUTPUT_ROOT']).expanduser().resolve() / 'current'
paths

In [ ]:
baseline = write_current_model_outputs(output_dir, paths)
pd.Series(baseline["metadata"], name="value").to_frame()

## Aggregate evaluation

Brier score and log loss measure probability quality (lower is better). AUC measures ranking. Coverage and selective error describe the subset the hard-label policy would accept; class certification still controls whether those decisions can be released.

In [ ]:
display(baseline["summary"].round(5))
display(baseline["reconciliation"].assign(delta=lambda x: x.recomputed - x.persisted))

In [ ]:
plot_data = baseline["summary"].set_index("strategy")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_data[["brier", "log_loss"]].plot.bar(ax=axes[0], rot=25, title="Probability loss (lower is better)")
plot_data[["coverage", "selective_error"]].plot.bar(ax=axes[1], rot=25, title="Decision coverage and error")
for axis in axes:
    axis.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

## Source-level behavior

A pooled score can hide failure in an individual source. These are encounter-held-out metrics, which are the most relevant existing locally supported reconstruction evaluation.

In [ ]:
source_metrics = baseline["by_source"].query("strategy == 'encounter'")[
    ["SOURCE", "INDEPENDENT_ENCOUNTER_N", "SRKW_N", "BRIER", "LOG_LOSS", "AUC", "COVERAGE", "ACCEPTED_ERROR"]
].sort_values("BRIER", ascending=False)
display(source_metrics.round(5))

## Interpretation

The current pooled binary model is strong on encounter-held-out ranking, but it is not production-certified. The purged stress test has no accepted cases, source-level errors are heterogeneous, and the fitted binary probabilities cannot represent Other ecotypes. The next notebook treats those as explicit experiment targets.